# PicoCal - Beat the BDT (notebook 12)

Goal: a transformer architecture whose R3 `sigma_eff` beats the **fair all-region BDT** on clean signal (kNN-25, energy cut 1-100 GeV, train all regions / test R3). GPU-resident tensors for speed.

## Architecture priors from the literature (reports/architecture-research-2026.json)

- **Energy-weighted SUM pooling** (EFN / Deep Sets) is the IRC-safe additive aggregation; mean pooling is not. [EFN 1810.05165; calo Deep Sets 2310.04442]
- **Residual / correction target**: predict a correction to the calibrated energy sum (CMS DeepSC). [2204.10277]
- **ParT pairwise bias** `SoftMax(QK'/sqrt d + U)V`. [2202.03772]
- Honest ceiling: on clean showers SOTA GNNs only *tie* the calibrated baseline; NN wins over BDT are modest and mostly appear under background. [2204.01681]

Variants: V0 mean - V1 mean+agg - V2 sum+agg - V3 energy-weighted-sum(EFN)+agg - V4 EFN+residual - V5 mean+residual.

In [1]:
import sys, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, split, resolution, EPS

FILES = 100
SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 150, "patience": 25}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "full").glob("matched_*.root"))[:FILES]

D = build(files, 3, 100.0, selector=lambda c: select_knn(c, 25))
toks = D["tok_seed"]; y = D["y"]; Et = D["Etrue"]; region = D["region"]; agg = D["agg"]
keep = (Et >= 1.0) & (Et <= 100.0)
ridx = np.flatnonzero((region == 3) & keep)
rtr, rva, rte = (ridx[s] for s in split(len(ridx)))
ttr = np.setdiff1d(np.setdiff1d(np.flatnonzero(keep), rte), rva)
in_dim = toks[int(ridx[0])].shape[1]

G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
n_global = G.shape[1]
la, lb = np.polyfit(agg[ttr, 0], y[ttr], 1)
base_all = (la * agg[:, 0] + lb).astype(np.float32)

gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[ttr], y[ttr])
BDT = float(resolution(np.exp(gb.predict(agg[rte])), Et[rte])["sigma_eff"])
ta, tb = np.polyfit(np.log(D["total_energy"][ttr] + EPS), y[ttr], 1)
TE = float(resolution(np.exp(ta * np.log(D["total_energy"][rte] + EPS) + tb), Et[rte])["sigma_eff"])
SUMcal = float(resolution(np.exp(la * agg[rte, 0] + lb), Et[rte])["sigma_eff"])

maxL = max(t.shape[0] for t in toks)
N = len(toks)
Xall = np.zeros((N, maxL, in_dim), np.float32)
Mall = np.zeros((N, maxL), np.bool_)
Wall = np.zeros((N, maxL), np.float32)
for i, t in enumerate(toks):
    L = t.shape[0]; Xall[i, :L] = t; Mall[i, :L] = True
    e = np.expm1(np.clip(t[:, 0], 0, None)); Wall[i, :L] = e / (e.sum() + 1e-9)
cont = np.concatenate([toks[i][:, :7] for i in ttr], 0); tmean = cont.mean(0); tstd = cont.std(0) + EPS
Xall[:, :, :7] = (Xall[:, :, :7] - tmean) / tstd
Xall[~Mall] = 0.0
gmean = G[ttr].mean(0); gstd = G[ttr].std(0) + EPS
Gall = ((G - gmean) / gstd).astype(np.float32)

Xt = torch.from_numpy(Xall).to(DEVICE); Mt = torch.from_numpy(Mall).to(DEVICE)
Wt = torch.from_numpy(Wall).to(DEVICE); Gt = torch.from_numpy(Gall).to(DEVICE)
Bt = torch.from_numpy(base_all).unsqueeze(1).to(DEVICE)
Yt = torch.from_numpy(y.astype(np.float32)).unsqueeze(1).to(DEVICE)
{"BDT_bar": round(BDT, 4), "total_energy": round(TE, 4), "sum_calib": round(SUMcal, 4),
 "n_train_all": int(len(ttr)), "n_test_R3": int(len(rte)), "device": DEVICE, "maxL": int(maxL)}

{'BDT_bar': 0.0388,
 'total_energy': 0.0635,
 'sum_calib': 0.0622,
 'n_train_all': 30061,
 'n_test_R3': 1706,
 'device': 'cuda',
 'maxL': 25}

In [2]:
def train_eval(make_model, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = make_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    def batches(idx, bs, shuffle):
        idx = np.asarray(idx)
        if shuffle:
            idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield Xt[b], Mt[b], Wt[b], Gt[b], Bt[b], Yt[b]

    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, w, g, base, yb in batches(rva, 512, False):
                s += nn.functional.mse_loss(model(X, m, w, g, base), yb).item(); c += 1
        return s / max(c, 1)

    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, w, g, base, yb in batches(ttr, cfg["batch"], True):
            opt.zero_grad()
            nn.functional.mse_loss(model(X, m, w, g, base), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4:
            best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(bstate); model.eval()

    def predict(idx):
        out = []
        with torch.no_grad():
            for X, m, w, g, base, _ in batches(idx, 512, False):
                out.append(model(X, m, w, g, base).cpu().numpy().ravel())
        return np.concatenate(out)

    pv, pt = predict(rva), predict(rte)
    a, b = np.polyfit(pv, y[rva], 1)
    return float(resolution(np.exp(a * pt + b), Et[rte])["sigma_eff"]), model

In [3]:
def encoder():
    layer = nn.TransformerEncoderLayer(cfg["d"], cfg["nhead"], dim_feedforward=4 * cfg["d"],
                                       dropout=cfg["dropout"], batch_first=True)
    return nn.TransformerEncoder(layer, cfg["layers"], enable_nested_tensor=False)

def head(nf):
    return nn.Sequential(nn.Linear(nf, cfg["d"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["d"], 1))

class Base(nn.Module):
    def __init__(self, extra=0):
        super().__init__()
        self.embed = nn.Linear(in_dim, cfg["d"]); self.enc = encoder()
        self.norm = nn.LayerNorm(cfg["d"]); self.head = head(cfg["d"] + extra)
    def encode(self, x, m):
        return self.enc(self.embed(x), src_key_padding_mask=~m)

class V0_mean(Base):
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        return self.head(self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1)))

class V1_hybrid(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return self.head(torch.cat([p, g], 1))

class V2_sumhybrid(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        return self.head(torch.cat([self.norm((h * wm).sum(1)), g], 1))

class V3_efn(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m)
        return self.head(torch.cat([self.norm((h * w.unsqueeze(-1)).sum(1)), g], 1))

class V4_efn_residual(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m)
        return base + self.head(torch.cat([self.norm((h * w.unsqueeze(-1)).sum(1)), g], 1))

class V5_mean_residual(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))

In [4]:
VARIANTS = {"V0_mean": V0_mean, "V1_hybrid": V1_hybrid, "V2_sumhybrid": V2_sumhybrid,
            "V3_efn": V3_efn, "V4_efn_residual": V4_efn_residual, "V5_mean_residual": V5_mean_residual}
res12 = {}; models = {}; rows = []
for name, cls in VARIANTS.items():
    vals = []
    for s in range(SEEDS):
        sig, mdl = train_eval(cls, s)
        vals.append(sig)
        if s == 0:
            models[name] = mdl
    mean, std = float(np.mean(vals)), float(np.std(vals))
    res12[name] = vals
    rows.append({"variant": name, "sigma_eff": round(mean, 4), "std": round(std, 4),
                 "BDT": round(BDT, 4), "beats_BDT": mean < BDT})
    print(f"{name}: {mean:.4f} +/- {std:.4f} {'BEATS' if mean < BDT else 'loses'} BDT {BDT:.4f}", flush=True)
summary12 = pd.DataFrame(rows).sort_values("sigma_eff").reset_index(drop=True)
summary12

V0_mean: 0.0493 +/- 0.0036 loses BDT 0.0388


V1_hybrid: 0.0479 +/- 0.0063 loses BDT 0.0388


V2_sumhybrid: 0.0482 +/- 0.0019 loses BDT 0.0388


V3_efn: 0.0453 +/- 0.0012 loses BDT 0.0388


V4_efn_residual: 0.0391 +/- 0.0044 loses BDT 0.0388


V5_mean_residual: 0.0360 +/- 0.0008 BEATS BDT 0.0388


,variant,sigma_eff,std,BDT,beats_BDT
0,V5_mean_residual,0.0360,0.0008,0.0388,True
1,V4_efn_residual,0.0391,0.0044,0.0388,False
2,V3_efn,0.0453,0.0012,0.0388,False
3,V1_hybrid,0.0479,0.0063,0.0388,False
4,V2_sumhybrid,0.0482,0.0019,0.0388,False
5,V0_mean,0.0493,0.0036,0.0388,False


In [5]:
import plotly.graph_objects as go
d = summary12
colors = ["#2ca02c" if b else "#8c8c8c" for b in d["beats_BDT"]]
fig = go.Figure(go.Bar(x=d["variant"], y=d["sigma_eff"], error_y=dict(type="data", array=d["std"]),
                       marker_color=colors, text=[f"{v:.4f}" for v in d["sigma_eff"]], textposition="outside"))
fig.add_hline(y=BDT, line_dash="dash", line_color="crimson", annotation_text=f"fair BDT {BDT:.4f}", annotation_position="top left")
fig.add_hline(y=SUMcal, line_dash="dot", line_color="darkorange", annotation_text=f"sum-calib {SUMcal:.4f}", annotation_position="bottom left")
fig.update_layout(template="plotly_white", height=450, title="R3 resolution vs fair BDT (green = beats BDT)",
                  yaxis_title="sigma_eff", xaxis_tickangle=-20)
fig.show()